# Fisher Information Overlap Independence Test (IF vs Math)

This notebook measures whether IF and Math tasks activate different parameter regions
in Fisher diagonal space.

## Method

Given Fisher diagonals `F_if` and `F_math`:

1. Build global scalar importance vectors over all parameters.
2. Extract global top-k% indices for each task.
3. Measure overlap:
   - `overlap = |TopK_if ∩ TopK_math| / k`
4. Compare with random baseline:
   - `random_baseline = k / N` (`N`: total scalar parameters)

If `overlap` is close to `random_baseline`, the two tasks are close to independent
in terms of Fisher-important parameter locations.

## Notes

- Default top-k percents are intentionally small to keep memory/runtime practical.
- The extraction is exact for the requested `top_k` under deterministic ordering.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Mapping, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


@dataclass(frozen=True)
class TaskFisherSpec:
    """Specification for one precomputed Fisher diagonal artifact.

    Args:
        name: Short task name used in logs/artifacts.
        fisher_path: Absolute path to serialized Fisher `.pt` dictionary.
    """

    name: str
    fisher_path: Path


@dataclass(frozen=True)
class OverlapRuntimeConfig:
    """Runtime settings controlling reproducibility and artifact outputs.

    Args:
        output_root: Directory where CSV/JSON/PNG artifacts are written.
        seed: Random seed for deterministic behavior.
    """

    output_root: Path
    seed: int = 7


@dataclass(frozen=True)
class OverlapStudyConfig:
    """Study configuration for top-k Fisher overlap analysis.

    Args:
        top_percent_values: Requested top-k percentages in `%` unit.
            Example: `0.01` means top `0.01%` of all scalar parameters.
        min_top_k: Lower bound for top-k size to avoid fragile tiny sets.
            The effective `k` is `max(ceil(N * p/100), min_top_k)`.
    """

    top_percent_values: Tuple[float, ...] = (0.0005, 0.001, 0.005, 0.01)
    min_top_k: int = 1024


IF_SPEC = TaskFisherSpec(
    name="if",
    fisher_path=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/fisher_diagonal/fisher_diag_if.pt"
    ),
)

MATH_SPEC = TaskFisherSpec(
    name="math",
    fisher_path=Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/fisher_diagonal/fisher_diag_math.pt"
    ),
)

RUNTIME = OverlapRuntimeConfig(
    output_root=Path("merging_analysis/artifacts/fisher_overlap_independence"),
    seed=7,
)

STUDY_CFG = OverlapStudyConfig(
    top_percent_values=(0.0005, 0.001, 0.005, 0.01),
    min_top_k=1024,
)


def set_seed(seed: int) -> None:
    """Set random seeds for reproducible analysis behavior.

    Args:
        seed: Integer random seed.

    Returns:
        None.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



def to_json_compatible(obj: Any) -> Any:
    """Recursively convert numpy/pandas scalars into JSON-compatible types.

    Args:
        obj: Arbitrary nested object.

    Returns:
        Object made of JSON-serializable Python primitives.
    """

    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [to_json_compatible(value) for value in obj]
    if isinstance(obj, tuple):
        return tuple(to_json_compatible(value) for value in obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    return obj



def load_fisher_dictionary(fisher_path: Path) -> Dict[str, torch.Tensor]:
    """Load a Fisher diagonal dictionary from disk.

    Why this function exists:
        The large `.pt` file must be validated early so downstream overlap code can
        assume a clean `Dict[str, Tensor]` structure with floating tensors on CPU.

    Args:
        fisher_path: Path to serialized Fisher dictionary.

    Returns:
        Mapping from parameter name to Fisher-diagonal tensor.

    Raises:
        FileNotFoundError: If `fisher_path` does not exist.
        TypeError: If object structure is not `Dict[str, Tensor]`.
        ValueError: If dictionary is empty.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f"Fisher artifact not found: {fisher_path}")

    fisher_obj = torch.load(fisher_path, map_location="cpu")

    if not isinstance(fisher_obj, dict):
        raise TypeError(
            f"Expected fisher object to be dict, but got {type(fisher_obj)} from {fisher_path}"
        )

    normalized: Dict[str, torch.Tensor] = {}

    for parameter_name, fisher_tensor in fisher_obj.items():
        if not isinstance(parameter_name, str):
            raise TypeError(f"Fisher key must be str, but got {type(parameter_name)}")
        if not isinstance(fisher_tensor, torch.Tensor):
            raise TypeError(
                f"Fisher value for '{parameter_name}' must be Tensor, got {type(fisher_tensor)}"
            )
        if not torch.is_floating_point(fisher_tensor):
            raise TypeError(
                f"Fisher tensor for '{parameter_name}' must be floating, got {fisher_tensor.dtype}"
            )

        # Keep tensors on CPU in float32 so arithmetic is stable and predictable.
        normalized[parameter_name] = fisher_tensor.detach().to(device="cpu", dtype=torch.float32)

    if not normalized:
        raise ValueError(f"Loaded empty fisher dictionary from {fisher_path}")

    return normalized



def build_parameter_offsets(
    fisher_dictionary: Mapping[str, torch.Tensor],
    parameter_order: Sequence[str],
) -> Tuple[Dict[str, int], int]:
    """Build deterministic global offsets for per-parameter flattened indices.

    Why this function exists:
        Top-k overlap must be measured in a shared global index space. Offsets let us
        map each local tensor index to a unique global scalar index.

    Args:
        fisher_dictionary: Fisher mapping for one reference task.
        parameter_order: Deterministic parameter ordering shared by both tasks.

    Returns:
        Tuple of:
        - `offsets`: parameter name -> starting global index.
        - `total_numel`: total scalar parameter count across all tensors.
    """

    offsets: Dict[str, int] = {}
    running_offset = 0

    for parameter_name in parameter_order:
        offsets[parameter_name] = int(running_offset)
        running_offset += int(fisher_dictionary[parameter_name].numel())

    return offsets, int(running_offset)



def validate_fisher_against_reference(
    fisher_dictionary: Mapping[str, torch.Tensor],
    reference_parameter_order: Sequence[str],
    reference_shapes: Mapping[str, Tuple[int, ...]],
) -> None:
    """Validate key and shape compatibility against a reference Fisher manifest.

    Args:
        fisher_dictionary: Candidate Fisher mapping to validate.
        reference_parameter_order: Ordered key list from reference task.
        reference_shapes: Reference parameter shape map.

    Returns:
        None. Raises if mismatch is detected.

    Raises:
        ValueError: If keys or shapes differ from the reference manifest.
    """

    candidate_keys = set(fisher_dictionary.keys())
    reference_keys = set(reference_parameter_order)

    if candidate_keys != reference_keys:
        missing = sorted(reference_keys - candidate_keys)
        extra = sorted(candidate_keys - reference_keys)
        raise ValueError(
            "Fisher key mismatch against reference manifest. "
            f"Missing={missing[:5]} (count={len(missing)}), "
            f"Extra={extra[:5]} (count={len(extra)})"
        )

    for parameter_name in reference_parameter_order:
        candidate_shape = tuple(fisher_dictionary[parameter_name].shape)
        reference_shape = tuple(reference_shapes[parameter_name])
        if candidate_shape != reference_shape:
            raise ValueError(
                f"Fisher shape mismatch at '{parameter_name}': "
                f"candidate={candidate_shape}, reference={reference_shape}"
            )



def resolve_topk_plan(
    total_numel: int,
    top_percent_values: Sequence[float],
    min_top_k: int,
) -> List[Dict[str, Any]]:
    """Resolve requested top-k percentages into executable integer `k` values.

    Args:
        total_numel: Total scalar parameter count (`N`).
        top_percent_values: Requested percentages in `%` unit.
        min_top_k: Minimum allowed top-k size.

    Returns:
        Ordered plan list where each item contains:
        - `requested_top_percent`
        - `effective_top_percent`
        - `top_k`
    """

    if total_numel <= 0:
        raise ValueError("total_numel must be positive")

    if min_top_k <= 0:
        raise ValueError("min_top_k must be positive")

    raw_plan: List[Dict[str, Any]] = []

    for requested_percent in top_percent_values:
        if requested_percent <= 0.0 or requested_percent >= 100.0:
            raise ValueError(
                "Each top percent must be in (0, 100). "
                f"Received {requested_percent}."
            )

        from_percent = int(math.ceil(float(total_numel) * (float(requested_percent) / 100.0)))
        effective_k = int(max(1, from_percent, int(min_top_k)))
        effective_k = int(min(effective_k, int(total_numel)))

        effective_percent = 100.0 * float(effective_k) / float(total_numel)

        raw_plan.append(
            {
                "requested_top_percent": float(requested_percent),
                "effective_top_percent": float(effective_percent),
                "top_k": int(effective_k),
            }
        )

    # Remove duplicate `k` values while preserving ascending-k order.
    deduped_plan: List[Dict[str, Any]] = []
    seen_k = set()
    for item in sorted(raw_plan, key=lambda row: int(row["top_k"])):
        k_value = int(item["top_k"])
        if k_value in seen_k:
            continue
        deduped_plan.append(item)
        seen_k.add(k_value)

    return deduped_plan



def extract_global_topk_indices(
    fisher_dictionary: Mapping[str, torch.Tensor],
    parameter_order: Sequence[str],
    parameter_offsets: Mapping[str, int],
    top_k: int,
    tqdm_desc: str,
) -> np.ndarray:
    """Extract deterministic global top-k Fisher indices without full flattening.

    Why this function exists:
        Full flattening of all parameters for a 1.7B model is memory-heavy. This
        implementation keeps only `O(top_k)` candidates while scanning tensors.

    How it works:
        1. For each parameter tensor, take local top-k candidates.
        2. Merge candidates with running global top-k pool.
        3. Keep best `top_k` by value, breaking ties by smaller global index.

    Args:
        fisher_dictionary: Fisher mapping for a single task.
        parameter_order: Deterministic parameter name order.
        parameter_offsets: Global base index for each parameter tensor.
        top_k: Number of global indices to keep.
        tqdm_desc: Progress label.

    Returns:
        `np.ndarray` of shape `[top_k]` containing global scalar indices ordered by
        descending Fisher value (tie-broken by ascending index).

    Raises:
        ValueError: If `top_k` is not positive.
        RuntimeError: If extracted index count is insufficient or duplicates exist.
    """

    if top_k <= 0:
        raise ValueError(f"top_k must be positive, got {top_k}")

    selected_values = np.empty((0,), dtype=np.float32)
    selected_indices = np.empty((0,), dtype=np.int64)

    for parameter_name in tqdm(parameter_order, desc=tqdm_desc):
        flat_tensor = fisher_dictionary[parameter_name].reshape(-1)
        local_numel = int(flat_tensor.numel())
        local_k = int(min(int(top_k), local_numel))

        if local_k == 0:
            continue

        # Local top-k reduces each parameter tensor to only relevant candidates.
        local_values, local_indices = torch.topk(
            flat_tensor,
            k=local_k,
            largest=True,
            sorted=False,
        )

        local_values_np = local_values.cpu().numpy().astype(np.float32, copy=False)
        local_indices_np = local_indices.cpu().numpy().astype(np.int64, copy=False)
        local_global_indices = local_indices_np + int(parameter_offsets[parameter_name])

        # Merge local candidates into the running global pool.
        if selected_values.size == 0:
            combined_values = local_values_np
            combined_indices = local_global_indices
        else:
            combined_values = np.concatenate((selected_values, local_values_np), axis=0)
            combined_indices = np.concatenate((selected_indices, local_global_indices), axis=0)

        if combined_values.size > top_k:
            # `argpartition` keeps only the numerically largest candidates fast.
            candidate_selector = np.argpartition(combined_values, -int(top_k))[-int(top_k):]
            candidate_values = combined_values[candidate_selector]
            candidate_indices = combined_indices[candidate_selector]

            # Stable deterministic ordering: value desc, then index asc.
            stable_order = np.lexsort((candidate_indices, -candidate_values))
            selected_values = candidate_values[stable_order]
            selected_indices = candidate_indices[stable_order]
        else:
            stable_order = np.lexsort((combined_indices, -combined_values))
            selected_values = combined_values[stable_order]
            selected_indices = combined_indices[stable_order]

    if selected_indices.size < int(top_k):
        raise RuntimeError(
            f"Global top-k extraction returned {selected_indices.size} < requested {top_k}"
        )

    selected_indices = selected_indices[: int(top_k)]

    if np.unique(selected_indices).size != selected_indices.size:
        raise RuntimeError("Global top-k indices contain duplicates, expected unique indices")

    return selected_indices



def compute_random_overlap_statistics(total_numel: int, top_k: int) -> Dict[str, float]:
    """Compute random-baseline overlap expectation and standard deviation.

    Why this function exists:
        Independence is judged by comparing observed overlap to random overlap under
        hypergeometric sampling.

    Args:
        total_numel: Total scalar parameter count (`N`).
        top_k: Top-k set size for each task.

    Returns:
        Dictionary with:
        - `random_baseline`: expected overlap ratio `k / N`
        - `std_overlap`: standard deviation of overlap ratio under random sampling
    """

    if total_numel <= 1:
        return {"random_baseline": 0.0, "std_overlap": 0.0}

    selection_ratio = float(top_k) / float(total_numel)

    # Hypergeometric variance for intersection size of two size-k subsets from N.
    variance_intersection = (
        float(top_k)
        * selection_ratio
        * (1.0 - selection_ratio)
        * ((float(total_numel - top_k)) / float(total_numel - 1))
    )

    std_intersection = float(math.sqrt(max(variance_intersection, 0.0)))
    std_overlap = float(std_intersection / max(float(top_k), 1.0))

    return {
        "random_baseline": float(selection_ratio),
        "std_overlap": float(std_overlap),
    }



def compute_overlap_dataframe(
    if_top_indices: np.ndarray,
    math_top_indices: np.ndarray,
    topk_plan: Sequence[Mapping[str, Any]],
    total_numel: int,
) -> pd.DataFrame:
    """Compute overlap metrics for each requested top-k setting.

    Args:
        if_top_indices: IF global top-max-k index array.
        math_top_indices: Math global top-max-k index array.
        topk_plan: Plan rows from `resolve_topk_plan`.
        total_numel: Total scalar parameter count.

    Returns:
        DataFrame containing overlap, random baseline, ratio-to-random, and z-score.
    """

    rows: List[Dict[str, Any]] = []

    for plan_row in topk_plan:
        top_k = int(plan_row["top_k"])

        if_prefix = np.sort(if_top_indices[:top_k])
        math_prefix = np.sort(math_top_indices[:top_k])

        # Both prefix arrays contain unique indices, so `assume_unique=True` is valid.
        intersection_count = int(
            np.intersect1d(if_prefix, math_prefix, assume_unique=True).size
        )

        observed_overlap = float(intersection_count / max(top_k, 1))

        random_stats = compute_random_overlap_statistics(total_numel=total_numel, top_k=top_k)
        random_baseline = float(random_stats["random_baseline"])
        std_overlap = float(random_stats["std_overlap"])

        overlap_over_random = (
            float(observed_overlap / random_baseline) if random_baseline > 0.0 else np.nan
        )

        z_score = (
            float((observed_overlap - random_baseline) / std_overlap)
            if std_overlap > 0.0
            else np.nan
        )

        rows.append(
            {
                "requested_top_percent": float(plan_row["requested_top_percent"]),
                "effective_top_percent": float(plan_row["effective_top_percent"]),
                "top_k": int(top_k),
                "intersection_count": int(intersection_count),
                "observed_overlap": float(observed_overlap),
                "random_baseline": float(random_baseline),
                "overlap_minus_random": float(observed_overlap - random_baseline),
                "overlap_over_random": float(overlap_over_random),
                "std_overlap_random": float(std_overlap),
                "z_score_vs_random": float(z_score),
                "near_random_2sigma": bool(abs(z_score) <= 2.0) if np.isfinite(z_score) else False,
            }
        )

    overlap_df = pd.DataFrame(rows)

    if not overlap_df.empty:
        overlap_df = overlap_df.sort_values("top_k").reset_index(drop=True)

    return overlap_df



def render_overlap_plot(overlap_df: pd.DataFrame) -> plt.Figure:
    """Render overlap-vs-random diagnostic plots.

    Args:
        overlap_df: DataFrame returned by `compute_overlap_dataframe`.

    Returns:
        Matplotlib figure object.
    """

    if overlap_df.empty:
        raise ValueError("overlap_df is empty; cannot render plot")

    x_values = overlap_df["effective_top_percent"].to_numpy(dtype=np.float64)

    figure, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(
        x_values,
        overlap_df["observed_overlap"].to_numpy(dtype=np.float64),
        marker="o",
        linewidth=2,
        label="Observed overlap",
    )
    axes[0].plot(
        x_values,
        overlap_df["random_baseline"].to_numpy(dtype=np.float64),
        marker="s",
        linewidth=2,
        linestyle="--",
        label="Random baseline (k / N)",
    )
    axes[0].set_title("Top-k overlap vs random baseline")
    axes[0].set_xlabel("Effective top-k (%)")
    axes[0].set_ylabel("Overlap ratio")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        x_values,
        overlap_df["overlap_over_random"].to_numpy(dtype=np.float64),
        marker="o",
        linewidth=2,
        label="Observed / Random",
    )
    axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Independence line (=1)")
    axes[1].set_title("Relative enrichment over random")
    axes[1].set_xlabel("Effective top-k (%)")
    axes[1].set_ylabel("Enrichment ratio")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    figure.tight_layout()

    return figure



def save_overlap_artifacts(
    output_root: Path,
    runtime_cfg: OverlapRuntimeConfig,
    study_cfg: OverlapStudyConfig,
    if_spec: TaskFisherSpec,
    math_spec: TaskFisherSpec,
    total_numel: int,
    topk_plan: Sequence[Mapping[str, Any]],
    overlap_df: pd.DataFrame,
    figure: plt.Figure,
) -> Dict[str, Path]:
    """Save overlap results to CSV/JSON/PNG artifact files.

    Args:
        output_root: Destination root directory.
        runtime_cfg: Runtime configuration dataclass.
        study_cfg: Study configuration dataclass.
        if_spec: IF Fisher input spec.
        math_spec: Math Fisher input spec.
        total_numel: Total scalar parameter count.
        topk_plan: Resolved top-k execution plan.
        overlap_df: Computed overlap dataframe.
        figure: Matplotlib figure for overlap diagnostics.

    Returns:
        Mapping of artifact labels to saved file paths.
    """

    output_root.mkdir(parents=True, exist_ok=True)

    csv_path = output_root / "scalar_topk_overlap_results.csv"
    json_path = output_root / "scalar_topk_overlap_summary.json"
    png_path = output_root / "scalar_topk_overlap_plot.png"

    overlap_df.to_csv(csv_path, index=False)
    figure.savefig(png_path, dpi=180, bbox_inches="tight")

    summary_payload = {
        "created_at": datetime.now().isoformat(),
        "runtime_config": asdict(runtime_cfg),
        "study_config": asdict(study_cfg),
        "if_spec": asdict(if_spec),
        "math_spec": asdict(math_spec),
        "total_numel": int(total_numel),
        "topk_plan": [dict(row) for row in topk_plan],
        "results": overlap_df.to_dict(orient="records"),
        "independence_criterion": "Observed overlap close to random baseline (k/N).",
    }

    with json_path.open("w", encoding="utf-8") as handle:
        json.dump(to_json_compatible(summary_payload), handle, indent=2)

    return {
        "csv": csv_path,
        "json": json_path,
        "png": png_path,
    }


set_seed(RUNTIME.seed)
RUNTIME.output_root.mkdir(parents=True, exist_ok=True)

print("Configured Fisher overlap study")
print(f"- IF Fisher path:   {IF_SPEC.fisher_path}")
print(f"- Math Fisher path: {MATH_SPEC.fisher_path}")
print(f"- Output root:      {RUNTIME.output_root}")
print(f"- Requested top percents (%): {STUDY_CFG.top_percent_values}")
print(f"- Minimum top-k: {STUDY_CFG.min_top_k}")


In [ ]:
# -----------------------------------------------------------------------------
# Step 1: Load IF Fisher and build the shared global index manifest.
# -----------------------------------------------------------------------------
if_fisher = load_fisher_dictionary(IF_SPEC.fisher_path)

parameter_order = sorted(if_fisher.keys())
reference_shapes = {name: tuple(if_fisher[name].shape) for name in parameter_order}
parameter_offsets, total_numel = build_parameter_offsets(
    fisher_dictionary=if_fisher,
    parameter_order=parameter_order,
)

print(f"Loaded IF Fisher tensors: {len(parameter_order)}")
print(f"Total scalar parameters (N): {total_numel:,}")

# Resolve top-k settings once using total parameter count.
topk_plan = resolve_topk_plan(
    total_numel=total_numel,
    top_percent_values=STUDY_CFG.top_percent_values,
    min_top_k=STUDY_CFG.min_top_k,
)

if not topk_plan:
    raise ValueError("topk_plan is empty; check STUDY_CFG values")

max_top_k = int(max(row["top_k"] for row in topk_plan))

print("Resolved top-k plan:")
print(pd.DataFrame(topk_plan).to_string(index=False))
print(f"Max top-k used for extraction: {max_top_k:,}")

# Extract IF top-max-k global indices, then free IF Fisher memory.
if_top_indices = extract_global_topk_indices(
    fisher_dictionary=if_fisher,
    parameter_order=parameter_order,
    parameter_offsets=parameter_offsets,
    top_k=max_top_k,
    tqdm_desc="Extract IF global top-k",
)

print(f"IF top-k index extraction complete: {if_top_indices.size:,} indices")

del if_fisher
gc.collect()

# -----------------------------------------------------------------------------
# Step 2: Load Math Fisher, validate compatibility, and extract top-k.
# -----------------------------------------------------------------------------
math_fisher = load_fisher_dictionary(MATH_SPEC.fisher_path)

validate_fisher_against_reference(
    fisher_dictionary=math_fisher,
    reference_parameter_order=parameter_order,
    reference_shapes=reference_shapes,
)

math_top_indices = extract_global_topk_indices(
    fisher_dictionary=math_fisher,
    parameter_order=parameter_order,
    parameter_offsets=parameter_offsets,
    top_k=max_top_k,
    tqdm_desc="Extract Math global top-k",
)

print(f"Math top-k index extraction complete: {math_top_indices.size:,} indices")

del math_fisher
gc.collect()

# -----------------------------------------------------------------------------
# Step 3: Compute overlap metrics and compare against random baseline.
# -----------------------------------------------------------------------------
overlap_df = compute_overlap_dataframe(
    if_top_indices=if_top_indices,
    math_top_indices=math_top_indices,
    topk_plan=topk_plan,
    total_numel=total_numel,
)

if overlap_df.empty:
    raise RuntimeError("No overlap rows were computed")

# Display core result table for quick interpretation.
display(
    overlap_df[
        [
            "requested_top_percent",
            "effective_top_percent",
            "top_k",
            "observed_overlap",
            "random_baseline",
            "overlap_over_random",
            "z_score_vs_random",
            "near_random_2sigma",
        ]
    ]
)

# -----------------------------------------------------------------------------
# Step 4: Render/save artifacts.
# -----------------------------------------------------------------------------
figure = render_overlap_plot(overlap_df)
artifact_paths = save_overlap_artifacts(
    output_root=RUNTIME.output_root,
    runtime_cfg=RUNTIME,
    study_cfg=STUDY_CFG,
    if_spec=IF_SPEC,
    math_spec=MATH_SPEC,
    total_numel=total_numel,
    topk_plan=topk_plan,
    overlap_df=overlap_df,
    figure=figure,
)

print("Saved artifacts:")
for artifact_name, artifact_path in artifact_paths.items():
    print(f"- {artifact_name}: {artifact_path}")

# Optional compact interpretation helper.
near_random_count = int(overlap_df["near_random_2sigma"].sum())
print(
    f"Near-random rows (|z| <= 2): {near_random_count}/{len(overlap_df)} "
    "(higher means stronger independence evidence)."
)


## Output Artifacts

After execution, artifacts are written to:

- `merging_analysis/artifacts/fisher_overlap_independence/scalar_topk_overlap_results.csv`
- `merging_analysis/artifacts/fisher_overlap_independence/scalar_topk_overlap_summary.json`
- `merging_analysis/artifacts/fisher_overlap_independence/scalar_topk_overlap_plot.png`

Interpretation guide:

- `observed_overlap` close to `random_baseline (k/N)` supports task independence.
- `overlap_over_random` close to `1.0` also indicates near-random overlap.
- `near_random_2sigma=True` means observed overlap is within 2-sigma of random expectation.
